In [19]:
import pandas as pd

In [20]:
df = pd.read_csv(r"C:\Users\user\Downloads\train_final_features_with_population (1).csv")

In [21]:
df.columns

Index(['sido', 'sigungu', 'eupmyeondong', 'stn', 'ABATT_DATE', 'JUDGE_DATE',
       'JUDGE_SEX', 'WEIGHT', 'BACKFAT', 'REA', 'WINDEX', 'WGRADE', 'INSFAT',
       'YUKSAK', 'FATSAK', 'TISSUE', 'GROWTH', 'AGE', 'BIRTH_YMD', 'CATTLE_NO',
       'FARM_UNIQUE_NO', 'LAST_GRADE', 'cold_tmin_m10_days_3m', 'wct_mean_3m',
       'thi_avg_mod_days_3m', 'thi_max_mod_days_3m', 'avg_population',
       'insfat_qgrade_rule_score', 'yuksak_qgrade_rule_score',
       'fatsak_qgrade_rule_score', 'tissue_qgrade_rule_score',
       'growth_downgrade_flag'],
      dtype='object')

In [22]:
kor_cols = {
    'sido': '시도',
    'sigungu': '시군구',
    'eupmyeondong': '읍면동',
    'stn': '기상관측소',
    'ABATT_DATE': '도축일자',
    'JUDGE_DATE': '판정일자',
    'JUDGE_SEX': '성별',
    'WEIGHT': '도체중',
    'BACKFAT': '등지방두께',
    'REA': '등심단면적',
    'WINDEX': '육량지수',
    'WGRADE': '육량등급',
    'INSFAT': '근내지방도',
    'YUKSAK': '육색',
    'FATSAK': '지방색',
    'TISSUE': '조직감',
    'GROWTH': '성숙도',
    'AGE': '나이',
    'BIRTH_YMD': '출생일자',
    'CATTLE_NO': '개체식별번호',
    'FARM_UNIQUE_NO': '농장고유번호',
    'LAST_GRADE': '최종등급',
    'cold_tmin_m10_days_3m': '최근3개월_최저기온_영하10도이하일수',
    'wct_mean_3m': '최근3개월_평균체감온도',
    'thi_avg_mod_days_3m': '최근3개월_평균THI_중등도일수',
    'thi_max_mod_days_3m': '최근3개월_최고THI_중등도일수',
    'avg_population': '평균인구수',
    'insfat_qgrade_rule_score': '근내지방도_품질등급_규칙점수',
    'yuksak_qgrade_rule_score': '육색_품질등급_규칙점수',
    'fatsak_qgrade_rule_score': '지방색_품질등급_규칙점수',
    'tissue_qgrade_rule_score': '조직감_품질등급_규칙점수',
    'growth_downgrade_flag': '성숙도_등급하락여부',
}

df = df.rename(columns=kor_cols)

In [24]:
df.columns

Index(['시도', '시군구', '읍면동', '기상관측소', '도축일자', '판정일자', '성별', '도체중', '등지방두께',
       '등심단면적', '육량지수', '육량등급', '근내지방도', '육색', '지방색', '조직감', '성숙도', '나이',
       '출생일자', '개체식별번호', '농장고유번호', '최종등급', '최근3개월_최저기온_영하10도이하일수',
       '최근3개월_평균체감온도', '최근3개월_평균THI_중등도일수', '최근3개월_최고THI_중등도일수', '평균인구수',
       '근내지방도_품질등급_규칙점수', '육색_품질등급_규칙점수', '지방색_품질등급_규칙점수', '조직감_품질등급_규칙점수',
       '성숙도_등급하락여부'],
      dtype='object')

In [25]:
df['육량등급'].unique()

array(['A', 'B', 'C'], dtype=object)

In [27]:
import time
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

from xgboost import XGBClassifier

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

RANDOM_SEED = 42
TARGET = "최종등급"

start_time = time.time()

print("[1/7] X, y 분리 시작", flush=True)

X = df.drop(columns=[TARGET])
y = df[TARGET]

target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)

print("[1/7] X, y 분리 완료", flush=True)

print("[2/7] train/valid/test 6:2:2 분할 시작", flush=True)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.4,
    random_state=RANDOM_SEED,
    stratify=y_encoded
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

print("Train:", X_train.shape, flush=True)
print("Valid:", X_valid.shape, flush=True)
print("Test :", X_test.shape, flush=True)

print("[2/7] 데이터 분할 완료", flush=True)

print("[3/7] 수치형 / 범주형 변수 구분", flush=True)

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("수치형 변수 개수:", len(numeric_features), flush=True)
print("범주형 변수 개수:", len(categorical_features), flush=True)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

print("[4/7] 전처리 fit_transform 시작", flush=True)

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

print("[4/7] 전처리 완료", flush=True)
print("전처리 후 Train shape:", X_train_processed.shape, flush=True)

results = []

print("\n[5/7] RandomForest 학습 시작", flush=True)

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    class_weight="balanced",
    verbose=1
)

rf_start = time.time()
rf_model.fit(X_train_processed, y_train)
rf_elapsed = time.time() - rf_start

print(f"[5/7] RandomForest 학습 완료: {rf_elapsed:.2f}초", flush=True)

print("[5/7] RandomForest 예측 및 평가 시작", flush=True)

rf_valid_pred = rf_model.predict(X_valid_processed)
rf_test_pred = rf_model.predict(X_test_processed)

results.append({
    "model": "RandomForest",
    "valid_accuracy": accuracy_score(y_valid, rf_valid_pred),
    "valid_f1_macro": f1_score(y_valid, rf_valid_pred, average="macro"),
    "test_accuracy": accuracy_score(y_test, rf_test_pred),
    "test_f1_macro": f1_score(y_test, rf_test_pred, average="macro")
})

print("\n===== RandomForest Test Classification Report =====")
print(classification_report(
    y_test,
    rf_test_pred,
    target_names=target_encoder.classes_
))

print("\n[6/7] XGBoost 학습 시작", flush=True)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=RANDOM_SEED,
    n_jobs=-1
)

xgb_start = time.time()
xgb_model.fit(
    X_train_processed,
    y_train,
    eval_set=[(X_valid_processed, y_valid)],
    verbose=25
)
xgb_elapsed = time.time() - xgb_start

print(f"[6/7] XGBoost 학습 완료: {xgb_elapsed:.2f}초", flush=True)

print("[6/7] XGBoost 예측 및 평가 시작", flush=True)

xgb_valid_pred = xgb_model.predict(X_valid_processed)
xgb_test_pred = xgb_model.predict(X_test_processed)

results.append({
    "model": "XGBoost",
    "valid_accuracy": accuracy_score(y_valid, xgb_valid_pred),
    "valid_f1_macro": f1_score(y_valid, xgb_valid_pred, average="macro"),
    "test_accuracy": accuracy_score(y_test, xgb_test_pred),
    "test_f1_macro": f1_score(y_test, xgb_test_pred, average="macro")
})

print("\n===== XGBoost Test Classification Report =====")
print(classification_report(
    y_test,
    xgb_test_pred,
    target_names=target_encoder.classes_
))

print("[7/7] 결과 정리 완료", flush=True)

results_df = pd.DataFrame(results)
display(results_df)

total_elapsed = time.time() - start_time
print(f"전체 소요 시간: {total_elapsed:.2f}초", flush=True)

[1/7] X, y 분리 시작
[1/7] X, y 분리 완료
[2/7] train/valid/test 6:2:2 분할 시작
Train: (1440292, 31)
Valid: (480098, 31)
Test : (480098, 31)
[2/7] 데이터 분할 완료
[3/7] 수치형 / 범주형 변수 구분
수치형 변수 개수: 21
범주형 변수 개수: 10
[4/7] 전처리 fit_transform 시작
[4/7] 전처리 완료
전처리 후 Train shape: (1440292, 1505849)

[5/7] RandomForest 학습 시작


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 22 concurrent workers.


KeyboardInterrupt: 

In [ ]:
def get_feature_names(fitted_pipeline):
    preprocessor = fitted_pipeline.named_steps["preprocess"]

    num_names = preprocessor.transformers_[0][2]

    cat_pipeline = preprocessor.named_transformers_["cat"]
    cat_encoder = cat_pipeline.named_steps["onehot"]
    cat_names = cat_encoder.get_feature_names_out(categorical_features)

    return list(num_names) + list(cat_names)


def plot_feature_importance(fitted_pipeline, title, top_n=20):
    feature_names = get_feature_names(fitted_pipeline)
    importances = fitted_pipeline.named_steps["model"].feature_importances_

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False).head(top_n)

    plt.figure(figsize=(10, 7))
    plt.barh(importance_df["feature"][::-1], importance_df["importance"][::-1])
    plt.title(title)
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()

    return importance_df


rf_importance = plot_feature_importance(
    rf_model,
    "RandomForest 변수중요도 Top 20"
)

xgb_importance = plot_feature_importance(
    xgb_model,
    "XGBoost 변수중요도 Top 20"
)

display(rf_importance)
display(xgb_importance)